# Our own DeBERTa member — with resume, and the fp32 instability fixed

Every DeBERTa we can attach is frozen and worth about **0.523** balanced accuracy on unseen prompts (our own
audit). A published 0.665 submission log replaced theirs with one fine-tuned on the clean pool (**0.5763**) and
gained on the leaderboard, and that pool is public. **435M parameters over the pool is about two GPU-hours**,
where an 8B LoRA is 211 hours per epoch.

**The first attempt diverged.** Loss went from 2.97 to `nan` by step ~200 — the classic DeBERTa-v3 fp32
instability — and the holdout read 0.0556, which is chance for 18 classes. Three fixes:

* **bf16 autocast** around the encoder forward. bf16 is the recommended precision for this architecture and
  needs no loss scaling;
* **the pooled vector divided by sqrt(hidden)** so a fresh head cannot overflow on it. The scale is folded
  back into the saved head weights, so the checkpoint keeps the plain `{weight, bias}` format;
* **a NaN guard**: a non-finite loss never touches the weights, and 200 of them abort the run instead of
  burning four hours of quota.

The pool loader also stopped at the first matching file and trained on 75k rows instead of all of them; it now
reads every source.

The output is the shape the inference notebook already loads — `encoder/`, `head.safetensors`, `meta.json`
with the label order — so the member drops in with **no new code path**. No per-layer norms here: that is the
LoRA24 (v5) format, not this one.

In [ ]:
%%writefile resume_lib.py
"""Resume support shared by both training notebooks.

Kaggle kills a GPU session at nine hours, and neither of our training jobs fits in one: the 8B member
needs about 5.2 hours for a 60k-row pass but a full-pool epoch is longer, and a 14B is longer still. So a
job has to survive being stopped and started again, or it never finishes at all.

The contract: the training kernel writes `resume_state.pt` next to its output every `SAVE_EVERY` steps.
The workflow publishes the whole output directory as a dataset, so the next session attaches that dataset
and picks up from the exact step — same sample order, same optimizer momentum, same RNG state.

Two details that are easy to get wrong and cost a whole session each:
  * the sample order must be **stored**, not regenerated — `np.random.default_rng(seed).permutation(n)` is
    stable across numpy versions but the *position within the epoch* is not, and a re-shuffled order means
    the resumed run sees different rows and the loss curve lies;
  * the optimizer state must be saved too — AdamW's moments are most of what makes a resumed run continue
    smoothly instead of re-warming for an hour.
"""
import os

import numpy as np
import torch


class Resume:
    def __init__(self, out_dir, input_dir, save_every=500, tag="train"):
        self.out_dir = out_dir
        self.input_dir = input_dir
        self.save_every = save_every
        self.tag = tag
        self.path = os.path.join(out_dir, "resume_state.pt")
        self.state = None

    # ------------------------------------------------------------------ find and load
    def find_published(self):
        """A previous session's state, published as a dataset and attached read-only."""
        import glob
        hits = sorted(glob.glob(f"{self.input_dir}/**/resume_state.pt", recursive=True))
        return hits[0] if hits else None

    def load(self):
        src = self.path if os.path.exists(self.path) else self.find_published()
        if src is None:
            print(f"[{self.tag}] no resume state — starting from step 0", flush=True)
            return None
        try:
            self.state = torch.load(src, map_location="cpu", weights_only=False)
        except Exception as e:
            print(f"[{self.tag}] resume state unreadable ({type(e).__name__}: {e}) — starting fresh", flush=True)
            return None
        print(f"[{self.tag}] RESUMING from {src}: step {self.state.get('step', 0):,}, "
              f"rows seen {self.state.get('seen', 0):,}, best BA {self.state.get('best_ba', -1):.4f}", flush=True)
        return self.state

    # ------------------------------------------------------------------ save
    def save(self, *, step, seen, best_ba, order, model, extra_tensors, optimizer=None, epoch=0):
        """Atomic write: a session killed mid-save must not leave a truncated file behind."""
        payload = {
            "step": int(step), "seen": int(seen), "best_ba": float(best_ba), "epoch": int(epoch),
            "order": np.asarray(order),
            "lora": {k: v.detach().cpu() for k, v in model.state_dict().items() if "lora_" in k},
            "extra": {k: (v.detach().cpu() if torch.is_tensor(v) else v) for k, v in extra_tensors.items()},
        }
        if optimizer is not None:
            payload["optimizer"] = optimizer.state_dict()
        tmp = self.path + ".tmp"
        torch.save(payload, tmp)
        os.replace(tmp, self.path)
        return self.path

    def restore_into(self, model, extra_modules, optimizer=None):
        """Put the loaded tensors back. Returns (step, seen, best_ba, order)."""
        if not self.state:
            return 0, 0, -1.0, None
        if self.state.get("lora"):
            missing, unexpected = model.load_state_dict(self.state["lora"], strict=False)
            print(f"[{self.tag}] restored {len(self.state['lora'])} LoRA tensors "
                  f"(missing {len(missing)}, unexpected {len(unexpected)})", flush=True)
        extra = self.state.get("extra", {}) or {}
        for name, mod in extra_modules.items():
            sd = extra.get(name)
            if sd:
                mod.load_state_dict(sd)
                print(f"[{self.tag}] restored {name} ({len(sd)} tensors)", flush=True)
            else:
                print(f"[{self.tag}] {name} not in the resume state — keeping its initial weights", flush=True)
        if optimizer is not None and "optimizer" in self.state:
            try:
                optimizer.load_state_dict(self.state["optimizer"])
                print(f"[{self.tag}] restored optimizer momentum", flush=True)
            except Exception as e:
                print(f"[{self.tag}] optimizer state not restored ({type(e).__name__}) — "
                      f"continuing with fresh momentum", flush=True)
        return (self.state.get("step", 0), self.state.get("seen", 0),
                float(self.state.get("best_ba", -1.0)), self.state.get("order"))


def make_order(n, seed):
    """One deterministic permutation for the whole run; stored so a resume sees the same rows."""
    return np.random.default_rng(seed).permutation(n)


def slice_from(order, start_step, bs, accum):
    """Where to begin inside the stored order for a resumed run."""
    per_step = bs * accum
    return int(start_step * per_step)

In [ ]:
"""Train our own DeBERTa member — the one training this budget can actually finish.

Every public DeBERTa we can attach is frozen and worth about 0.523 balanced accuracy on unseen prompts.
The 0.665 submission log replaced it with one fine-tuned on the clean pool (0.5763) and gained on the LB,
and the pool is public. The difference between that and the 8B LoRA is cost: 435M parameters over ~228k
passages is two GPU-hours on 2xT4, where an 8B LoRA is 211 hours per epoch and therefore impossible here.

We train in exactly the format the inference notebook already loads — `encoder/`, `head.safetensors`,
`meta.json` with the label order — so the member drops in with the existing loader and no new code path.

Metric note: the competition scores balanced accuracy, so the loss is class-weighted by inverse frequency;
plain cross-entropy on a pool where the classes run 20,579 to 6,594 would trade rare-class recall away.
"""
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

T0 = time.time()
INPUT_DIR = "/kaggle/input"
OUT = "/kaggle/working/deberta_ours"
os.makedirs(OUT, exist_ok=True)

CFG = {
    "epochs": 2,
    "bs": 16,
    "max_len": 256,
    "lr_encoder": 1e-5,
    "lr_head": 1e-4,
    "warmup": 0.06,
    "val_frac": 0.15,          # prompt-grouped holdout from the official 2,699 rows
    "pool_cap_per_class": 6000,  # caps the biggest class so the epoch fits the budget
    "seed": 7,
}
print("CONFIG:", json.dumps(CFG), flush=True)


def find(pat):
    hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pat}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"attach {pat}")
    return hits[0]


DATA = next(p for p in [f"{INPUT_DIR}/competitions/name-that-llm", f"{INPUT_DIR}/name-that-llm"]
            if os.path.exists(f"{p}/train.csv"))
train = pd.read_csv(f"{DATA}/train.csv")
LABELS = sorted(train.model.unique().tolist())
K = len(LABELS)
print(f"official train {len(train):,} rows, {train.prompt_id.nunique()} prompts, {K} labels", flush=True)

## Every pool source, capped per class; prompt-grouped holdout

In [ ]:
"""Pool rows: the merged corpus plus the api-gen and google-gen sets we already attach."""
frames = []
_seen = set()
for pat, txtcol in (("ntl_merged*.parquet", "text"), ("merged*.parquet", "text"),
                    ("*api_gen*.parquet", "text"), ("*google_gen*.parquet", "text"),
                    ("*google*gen*.parquet", "text"), ("pool*.parquet", "text")):
    for p in sorted(glob.glob(f"{INPUT_DIR}/**/{pat}", recursive=True)):
        if p in _seen:          # the patterns overlap; read each file once
            continue
        _seen.add(p)
        try:
            d = pd.read_parquet(p)
        except Exception as e:
            print("skip", p, type(e).__name__, flush=True)
            continue
        cols = {c.lower(): c for c in d.columns}
        t = cols.get("text") or cols.get(txtcol)
        m = cols.get("model") or cols.get("label")
        if t is None or m is None:
            continue
        d = d[[t, m]].rename(columns={t: "text", m: "model"})
        frames.append(d)
        print(f"  {p.split('/')[-1]:<44} {len(d):>9,} rows", flush=True)

print(f"read {len(frames)} pool files", flush=True)
pool = pd.concat(frames, ignore_index=True)
pool = pool[pool.model.isin(LABELS)]
pool["text"] = pool.text.astype(str)
pool = pool[pool.text.str.split().str.len().between(30, 300)]
pool = pool.drop_duplicates(subset=["text"])
print(f"pool after filter/dedup: {len(pool):,} rows", flush=True)

rng = np.random.default_rng(CFG["seed"])
capped = [g.sample(min(len(g), CFG["pool_cap_per_class"]), random_state=CFG["seed"])
          for _, g in pool.groupby("model")]
pool = pd.concat(capped, ignore_index=True).sample(frac=1.0, random_state=CFG["seed"])
print(f"pool used: {len(pool):,} rows, per class "
      f"{pool.model.value_counts().min():,}-{pool.model.value_counts().max():,}", flush=True)

# prompt-grouped holdout from the OFFICIAL rows only: the pool has no competition prompts, so training on
# it and validating on the official holdout is a genuine unseen-prompt measurement.
prompts = np.array(sorted(train.prompt_id.unique()))
rng.shuffle(prompts)
n_val = max(1, int(len(prompts) * CFG["val_frac"]))
val_p = set(prompts[:n_val].tolist())
va = train[train.prompt_id.isin(val_p)].reset_index(drop=True)
tr_official = train[~train.prompt_id.isin(val_p)].reset_index(drop=True)
print(f"val {len(va):,} rows / {va.prompt_id.nunique()} prompts | "
      f"official-train {len(tr_official):,} rows (folded into the pool)", flush=True)

Xtr = pd.concat([pool[["text", "model"]], tr_official[["text", "model"]]], ignore_index=True)
Xtr = Xtr.sample(frac=1.0, random_state=CFG["seed"]).reset_index(drop=True)
print(f"training rows: {len(Xtr):,}", flush=True)

## Encoder + head, bf16 autocast, scaled pooled features

In [ ]:
"""Model: the same encoder + pooled-linear-head shape the inference notebook already loads."""
from transformers import AutoConfig, AutoModel, AutoTokenizer

CKPT = os.path.dirname(find("deberta_v3_large/meta.json"))
enc_dir = os.path.join(CKPT, "encoder")
print("encoder from", enc_dir, flush=True)
tok = AutoTokenizer.from_pretrained(enc_dir)
config = AutoConfig.from_pretrained(enc_dir)
config.hidden_dropout_prob = 0.1
enc = AutoModel.from_pretrained(enc_dir, config=config).to("cuda:0")
H = enc.config.hidden_size
head = nn.Linear(H, K).to("cuda:0")

n_par = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in head.parameters())
print(f"trainable parameters: {n_par:,} ({n_par/1e6:.0f}M)", flush=True)

lab2i = {m: i for i, m in enumerate(LABELS)}
counts = Xtr.model.map(lab2i).value_counts().reindex(range(K), fill_value=0).to_numpy(np.float64)
w_cls = (counts.sum() / (K * np.clip(counts, 1, None))) ** 0.5      # sqrt-inverse frequency: softer than 1/n
w_cls = torch.tensor(w_cls / w_cls.mean(), dtype=torch.float32, device="cuda:0")
print("class weights (min/median/max):",
      f"{w_cls.min():.2f}/{w_cls.median():.2f}/{w_cls.max():.2f}", flush=True)

opt = torch.optim.AdamW([
    {"params": enc.parameters(), "lr": CFG["lr_encoder"]},
    {"params": head.parameters(), "lr": CFG["lr_head"]},
], weight_decay=0.01)
steps_per_epoch = int(np.ceil(len(Xtr) / CFG["bs"]))
total_steps = steps_per_epoch * CFG["epochs"]
sched = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=[CFG["lr_encoder"], CFG["lr_head"]], total_steps=total_steps,
    pct_start=CFG["warmup"], anneal_strategy="linear")
print(f"steps/epoch {steps_per_epoch:,} | total {total_steps:,}", flush=True)


def encode(texts, bs=None):
    """ids only; padding happens per batch so a short batch does not pay for a long one."""
    out = []
    bs = bs or 4096
    for s in range(0, len(texts), bs):
        out.extend(tok(list(texts[s:s + bs]), truncation=True, max_length=CFG["max_len"])["input_ids"])
    return out


FEAT_SCALE = float(np.sqrt(H))     # keeps the pooled vector near unit scale so the head cannot overflow


def pooled_fp32(bt):
    """Masked mean over the last hidden state, computed in bf16 and returned in fp32.

    DeBERTa-v3-large fine-tuned in plain fp32 went NaN at about step 200 on this data (loss 2.97 -> nan).
    bf16 autocast is the recommended precision for this architecture and does not need loss scaling, and
    dividing by sqrt(hidden) stops the pooled vector from being large enough to blow up the fresh head.
    """
    m = bt["attention_mask"].unsqueeze(-1).float()
    with torch.autocast("cuda", dtype=torch.bfloat16):
        h = enc(**bt).last_hidden_state
    pooled = (h.float() * m).sum(1) / m.sum(1).clamp(min=1)
    return pooled / FEAT_SCALE


@torch.inference_mode()
def evaluate(df):
    enc.eval()
    ids = encode(df.text.tolist())
    preds = np.zeros(len(ids), np.int64)
    for s in range(0, len(ids), 64):
        bt = tok.pad({"input_ids": ids[s:s + 64]}, return_tensors="pt").to("cuda:0")
        preds[s:s + 64] = head(pooled_fp32(bt)).argmax(1).cpu().numpy()
    enc.train()
    true = df.model.map(lab2i).to_numpy()
    per = [(preds[true == c] == c).mean() for c in range(K) if (true == c).any()]
    return float(np.mean(per)), preds, true

## Train with per-epoch resume and a NaN guard

In [ ]:
"""Train, tracking balanced accuracy on the unseen-prompt holdout after every epoch."""
from resume_lib import Resume, make_order
enc.train()
ids_all = encode(Xtr.text.tolist())
labels_all = Xtr.model.map(lab2i).to_numpy(np.int64)
print(f"tokenised {len(ids_all):,} rows in {time.time()-T0:.0f}s", flush=True)

# Resume: Kaggle stops a session at nine hours. This job fits in one, but a bigger pool or a slower GPU
# draw would not, so the state is written after every epoch and picked up from the attached dataset.
rs = Resume(OUT, INPUT_DIR, tag="deb")
best = {"ba": -1.0, "epoch": -1}
order_all = make_order(len(ids_all), CFG["seed"] + 1)
start_epoch = 0
state = rs.load()
if state:
    best["ba"] = float(state.get("best_ba", -1.0))
    start_epoch = int(state.get("epoch", 0))
    prev = state.get("extra", {}).get("encoder")
    if prev:
        enc.load_state_dict(prev)
        print(f"[deb] encoder restored from epoch {start_epoch}", flush=True)
    sd = state.get("extra", {}).get("head")
    if sd:
        head.load_state_dict(sd)
    if "optimizer" in state:
        try:
            opt.load_state_dict(state["optimizer"])
            print("[deb] optimizer momentum restored", flush=True)
        except Exception as e:
            print(f"[deb] optimizer not restored ({type(e).__name__})", flush=True)

for epoch in range(start_epoch, CFG["epochs"]):
    rng_t = np.random.default_rng(CFG["seed"] + 1 + epoch)
    order = rng_t.permutation(len(ids_all))
    run_loss, seen, skipped, step = 0.0, 0, 0, 0
    t_ep = time.time()
    for s in range(0, len(order), CFG["bs"]):
        idx = order[s:s + CFG["bs"]]
        if len(idx) < 2:
            continue
        bt = tok.pad({"input_ids": [ids_all[i] for i in idx]}, return_tensors="pt").to("cuda:0")
        logits = head(pooled_fp32(bt))
        yb = torch.from_numpy(labels_all[idx]).to("cuda:0")
        loss = F.cross_entropy(logits.float(), yb, weight=w_cls, label_smoothing=0.05)
        if not torch.isfinite(loss):
            # never let a bad batch touch the weights: skip it, and stop the run if it keeps happening
            skipped += 1
            opt.zero_grad(set_to_none=True)
            if skipped <= 5 or skipped % 50 == 0:
                print(f"  step {step}: non-finite loss, batch skipped ({skipped} so far)", flush=True)
            if skipped > 200:
                print(f"ABORT: {skipped} non-finite losses — the run is diverged, stopping", flush=True)
                break
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(list(enc.parameters()) + list(head.parameters()), 1.0)
        opt.step()
        sched.step()
        opt.zero_grad(set_to_none=True)
        run_loss += float(loss.detach()) * len(idx)
        seen += len(idx)
        step += 1
        if (s // CFG["bs"]) % 200 == 0:
            print(f"  ep{epoch+1} {seen:>7,}/{len(order):,} loss {run_loss/max(seen,1):.4f} "
                  f"({time.time()-t_ep:.0f}s)", flush=True)
    ba, _, _ = evaluate(va)
    print(f"EPOCH {epoch+1}: loss {run_loss/max(seen,1):.4f} | HOLDOUT BA {ba:.4f} "
          f"({time.time()-t_ep:.0f}s)", flush=True)
    if ba > best["ba"]:
        best = {"ba": ba, "epoch": epoch + 1}
        enc.save_pretrained(os.path.join(OUT, "encoder"))
        tok.save_pretrained(os.path.join(OUT, "encoder"))
        torch.save({"weight": head.weight.detach().cpu(), "bias": head.bias.detach().cpu()},
                   os.path.join(OUT, "head.pt"))
        print(f"  saved (best so far, BA {ba:.4f})", flush=True)
    # write the resume state after every epoch, so a session that dies here loses at most one epoch
    rs.save(step=epoch + 1, seen=(epoch + 1) * len(ids_all), best_ba=best["ba"], order=order_all,
            model=enc, extra_tensors={"encoder": enc.state_dict(), "head": head.state_dict()},
            optimizer=opt, epoch=epoch + 1)
    print(f"  resume state written (next session starts at epoch {epoch+1})", flush=True)

print(f"\nBEST HOLDOUT BA {best['ba']:.4f} at epoch {best['epoch']}", flush=True)

## Write the drop-in checkpoint

In [ ]:
"""Write the checkpoint in the exact shape the inference notebook loads, and record the measurement."""
from safetensors.torch import save_file

ck = torch.load(os.path.join(OUT, "head.pt"), map_location="cpu")
# We trained on pooled / FEAT_SCALE, but the inference cell applies the head to the raw pooled vector.
# Folding the scale into the weights keeps the checkpoint in the plain {weight, bias} format it expects:
#     (pooled / s) @ W.T + b  ==  pooled @ (W / s).T + b
save_file({"weight": (ck["weight"].float() / FEAT_SCALE), "bias": ck["bias"].float()},
          os.path.join(OUT, "head.safetensors"))
os.remove(os.path.join(OUT, "head.pt"))

meta = {
    "labels": LABELS,                       # the inference cell asserts this equals its own LABELS
    "max_len": CFG["max_len"],
    "hidden_size": H,
    "pooling": "attention-masked mean of last_hidden_state; sqrt(hidden) scale folded into the head",
    "feat_scale": float(FEAT_SCALE),
    "trained_on": {
        "pool_rows": int(len(pool)),
        "official_rows": int(len(tr_official)),
        "total": int(len(Xtr)),
        "val_frac": CFG["val_frac"],
        "epochs": CFG["epochs"],
        "lr_encoder": CFG["lr_encoder"],
        "lr_head": CFG["lr_head"],
        "class_weight": "sqrt inverse frequency",
        "label_smoothing": 0.05,
        "seed": CFG["seed"],
    },
    "holdout_ba": round(float(best["ba"]), 4),
    "holdout_epoch": int(best["epoch"]),
    "holdout_prompts": int(va.prompt_id.nunique()),
    "holdout_rows": int(len(va)),
    "baseline_reference": {
        "note": "public frozen deberta measured 0.5230 unseen-prompt BA in our own audit",
        "published_pool_deberta": 0.5763,
    },
    "trained_in_seconds": round(time.time() - T0),
}
json.dump(meta, open(os.path.join(OUT, "meta.json"), "w"), indent=2)

print("=== deb_ours ===", flush=True)
print(json.dumps({k: meta[k] for k in ("holdout_ba", "holdout_epoch", "holdout_prompts", "trained_in_seconds")},
                 indent=1), flush=True)
print("files:", sorted(os.listdir(OUT)), flush=True)
print(f"encoder size {sum(os.path.getsize(os.path.join(OUT,'encoder',f)) for f in os.listdir(os.path.join(OUT,'encoder')))/2**20:.0f} MB",
      flush=True)
print("DROP-IN READY: attach this dataset and the existing deberta cell loads it unchanged", flush=True)